In [15]:
from pathlib import Path
import sys

# Get the project root (one level up from notebooks/)
project_root = Path().resolve().parent

# Add project root to Python path
sys.path.append(str(project_root))

# Optional: create __init__.py in all packages (if missing)
folders = ["preprocessing", "models", "training", "evaluation", "generation"]
for f in folders:
    folder_path = project_root / f
    folder_path.mkdir(exist_ok=True)  # create folder if it doesn't exist
    init_file = folder_path / "__init__.py"
    if not init_file.exists():
        init_file.touch()

### *Step 1: Imports*

*Import necessary modules and functions. `data_utils` contains preprocessing functions, `model` defines our neural network, a   `train_utils` handles the training loop, `test_utils` handles testing, and `predict` provides prediction functionality.*


In [16]:
from pathlib import Path

# Data utils
from preprocessing.data_utils import load_and_tokenize, build_vocab, prepare_data

# Model
from models.model import FeedforwardNN

# Training & testing
from training.train_utils import train_model
from evaluation.test_utils import test_model

# Prediction & generation
from generation.predict import predict_next_word, generate_sentence

# Evaluation
from evaluation.metrics import calculate_perplexity, calculate_bleu
from evaluation.plots import plot_loss

# PyTorch
import torch


### *Step 2: Load Dataset*

*Specify the path to the Penn Treebank dataset and load the training and validation sentences using our `load_and_tokenize` function.*

In [17]:
train_sentences = load_and_tokenize(r"../data/ptdataset/ptb.train.txt")
val_sentences   = load_and_tokenize(r"../data/ptdataset/ptb.valid.txt")
test_sentences  = load_and_tokenize(r"../data/ptdataset/ptb.test.txt")

print(f"Training sentences: {len(train_sentences)}")
print(f"Validation sentences: {len(val_sentences)}")
print(f"Test sentences: {len(test_sentences)}")


Training sentences: 42068
Validation sentences: 3370
Test sentences: 3761


### *Step 3: Build Vocabulary and Prepare Data*

*Create word-to-index and index-to-word mappings using `build_vocab`. Then, `prepare_data` converts tokenized sentences into numerical sequences for model training. Inputs are sequences of words, and outputs are the next words.*

In [18]:
word_to_index, index_to_word = build_vocab(train_sentences)

# Convert sentences → sequences
train_inputs, train_outputs, max_seq_len = prepare_data(train_sentences, word_to_index)
val_inputs, val_outputs, _ = prepare_data(val_sentences, word_to_index, max_len=max_seq_len)
test_inputs, test_outputs, _ = prepare_data(test_sentences, word_to_index, max_len=max_seq_len)

seq_len = max_seq_len - 1
vocab_size = len(word_to_index)

print(f"Vocab size: {vocab_size}, Max sequence length: {seq_len}")


Vocab size: 10007, Max sequence length: 116


### *Step 4: Initialize Model*

*Define the neural network with input size equal to the sequence length, two hidden layers (128 and 64 neurons), and output size equal to the vocabulary size. We then create an instance of `FeedforwardNN`.*

In [19]:
embedding_dim = 50
hidden_size1, hidden_size2 = 128, 64

model = FeedforwardNN(
    seq_len=seq_len,
    embedding_dim=embedding_dim,
    hidden1=hidden_size1,
    hidden2=hidden_size2,
    vocab_size=vocab_size
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

### *Step 5: Train the Model*

*Train the neural network using `train_model`, which handles batching, loss computation, backpropagation, and validation. For simplicity, we predict only the first next word in each sequence.*


In [20]:
# Modified train_model to return train/validation losses
train_losses, val_losses = train_model(
    model,
    train_inputs, train_outputs,
    val_inputs, val_outputs,
    batch_size=64,
    epochs=5,
    lr=1e-3,
    device=device
)

Epoch 1/5, Train Loss: 5.0461
Epoch 1/5, Validation Loss: 3.6760
Epoch 2/5, Train Loss: 2.5051
Epoch 2/5, Validation Loss: 2.7291
Epoch 3/5, Train Loss: 1.1664
Epoch 3/5, Validation Loss: 2.3007
Epoch 4/5, Train Loss: 0.4547
Epoch 4/5, Validation Loss: 2.2897
Epoch 5/5, Train Loss: 0.1491
Epoch 5/5, Validation Loss: 2.5416


###  *Step 6: Plot Loss Curves*

In [21]:
plot_loss(train_losses, val_losses, out_path="evaluation/plots/loss_curve.png")

### *Step 7: Evaluate on Test Set*

In [22]:
test_loss = test_model(model, test_inputs, test_outputs, device=device)
test_perplexity = calculate_perplexity(test_loss)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Perplexity: {test_perplexity:.4f}")


Test Loss: 1.8555
Test Perplexity: 6.3950


### *Step 8: Generate Sentences*

In [23]:
# Use first training sentence as seed
seed_seq = train_inputs[0]
seed_words = [index_to_word[idx] for idx in seed_seq[:3]]

generated_text = generate_sentence(
    model=model,
    seed_text=seed_words,
    word_to_index=word_to_index,
    index_to_word=index_to_word,
    max_seq_len=seq_len,
    max_gen_len=20,
    device=device
)
print("Generated sentence:", generated_text)

Generated sentence: aer banknote berlitz


### *Step 9: Compute BLEU Score*

In [24]:
# Using first test sentence as reference
reference_sentence = " ".join(test_sentences[0])
bleu_score = calculate_bleu(reference_sentence, generated_text)

print(f"BLEU score (first test sentence): {bleu_score:.4f}")

BLEU score (first test sentence): 0.0000


### *Step 6: Save the trained model*

In [25]:
torch.save(model.state_dict(), "model.pth")